# LIMIT — Multi-Model Evaluation

In [ ]:
from evaluate import eval_item_retrieval
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from dotenv import load_dotenv
load_dotenv()

In [ ]:
MODELS: dict[str, dict] = {
    # ~0.1 GB; fast baseline, decent MTEB score
    "BGE-small": {
        "hf_id": "BAAI/bge-small-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
    },
    # ~1.3 GB; strong MTEB ~2023, good general baseline
    "BGE-large": {
        "hf_id": "BAAI/bge-large-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
    },
    # ~0.5 GB; ModernBERT backbone, strong MTEB for its size (2024)
    "GTE-ModernBERT": {
        "hf_id": "Alibaba-NLP/gte-modernbert-base",
        "query_prefix": "query: ",
    },
    # ~2 GB; near SOTA on MTEB at release (2024)
    "Snowflake-Arctic-L": {
        "hf_id": "Snowflake/snowflake-arctic-embed-l",
        "query_prefix": "",
    },
    # ~1.2 GB; SOTA-tier small model, strong MTEB for size (2025)
    "Qwen3-Embedding-0.6B": {
        "hf_id": "Qwen/Qwen3-Embedding-0.6B",
        "query_prefix": "query: ",
    },
    # ~8 GB; near SOTA on MTEB (2025)
    "Qwen3-Embedding-4B": {
        "hf_id": "Qwen/Qwen3-Embedding-4B",
        "query_prefix": "query: ",
    },
    # ~16 GB; SOTA on MTEB at release (2025)
    "Qwen3-Embedding-8B": {
        "hf_id": "Qwen/Qwen3-Embedding-8B",
        "query_prefix": "query: ",
    },
    # ~14 GB; was SOTA on MTEB at release (2023)
    "E5-Mistral-7B": {
        "hf_id": "intfloat/e5-mistral-7b-instruct",
        "query_prefix": "Instruct: Retrieve the person whose profile contains the queried item.\nQuery: ",
    },
    # ~14 GB; unified embedding+generation, strong MTEB (2024)
    "GritLM-7B": {
        "hf_id": "GritLM/GritLM-7B",
        "query_prefix": "<|user|>\nRetrieve the person whose profile contains the queried item.\n<|embed|>\n",
    },
    # ~16 GB; instruction-following retrieval based on Llama 3.1 (2024)
    "Promptriever-Llama3-8B": {
        "hf_id": "samaya-ai/promptriever-llama3.1-8b-instruct-v1",
        "query_prefix": "Retrieve the person whose profile contains the queried item.\n\n",
    },
    # ~16 GB; #1 MTEB late 2024, instruction-following with latent attention layer
    "NV-Embed-v2": {
        "hf_id": "nvidia/NV-Embed-v2",
        "query_prefix": "Instruct: Retrieve the person whose profile contains the queried item.\nQuery: ",
    },
    # ~14 GB; strong MTEB 2024, instruction-following, Qwen2 backbone
    "GTE-Qwen2-7B": {
        "hf_id": "Alibaba-NLP/gte-Qwen2-7B-instruct",
        "query_prefix": "Instruct: Retrieve the person whose profile contains the queried item.\nQuery: ",
    },
    # ~1 GB; MoE architecture, 475M params (305M active), strong for size (2024)
    "Nomic-Embed-v2-MoE": {
        "hf_id": "nomic-ai/nomic-embed-text-v2-moe",
        "query_prefix": "search_query: ",
    },
    # ~2 GB; updated Arctic-L, stronger MTEB than v1 (2024)
    "Snowflake-Arctic-L-v2": {
        "hf_id": "Snowflake/snowflake-arctic-embed-l-v2.0",
        "query_prefix": "query: ",
    },
}

In [ ]:
MODEL_NAME = "GTE-ModernBERT"  # change to any key in MODELS

In [ ]:
N = 100
if "all_results" not in globals():
    all_results = {}

cfg = MODELS[MODEL_NAME]
print(f"\n{'='*60}\n{MODEL_NAME}")
all_results[MODEL_NAME] = eval_item_retrieval(
    n=N,
    model_name=cfg["hf_id"],
    query_prefix=cfg["query_prefix"],
)

In [ ]:
def plot_model_comparison(all_results, metric="recall@1"):
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = cm.tab10.colors
    for i, (label, results) in enumerate(all_results.items()):
        ms = sorted(results)
        scores = [results[m][metric] for m in ms]
        ax.plot(ms, scores, marker="o", label=label, color=colors[i % len(colors)])
    ax.set_xlabel("m (items per person)")
    ax.set_ylabel(metric)
    ax.set_title(f"Model comparison — {metric}")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_model_comparison(all_results, "recall@1")
plot_model_comparison(all_results, "mrr")